# Phishing Email Detection using a Bidirectional LSTM

**ITRI625 Project - Machine Learning / Deep Learning**

**Scenario:** A model that classifies emails as *phishing* or *legitimate* using their text content. **Dataset** used is `phishing_email.csv`, a combined, cleaned corpus built from several well-known public phishing/spam email sources (CEAS_08, Enron, Ling, Nazario, Nigerian Fraud, SpamAssassin). Each row contains the combined subject+body text (`text_combined`) and a binary `label` (1 = phishing, 0 = legitimate). **Approach** used is a Bidirectional LSTM text classifier trained on tokenized/padded email text, with per-epoch metric logging, early stopping (patience) and full evaluation (ROC, PR curve, confusion matrix).


## 1. Imports

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import pickle
import json

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

## 2. Load and Explore the Dataset

We use `phishing_email.csv`, a pre-combined and cleaned dataset built from several public phishing/legitimate
email corpora (CEAS_08, Enron, Ling, Nazario, Nigerian Fraud, SpamAssassin). Each row has:
- `text_combined`: the email's subject + body text
- `label`: 1 = phishing, 0 = legitimate (safe)


In [ ]:

df = pd.read_csv("phishing_email.csv")
print("Shape:", df.shape)
df.head()


In [ ]:

# Check for missing values and duplicates
print("Missing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Drop any missing/empty text rows and duplicates
df = df.dropna(subset=["text_combined", "label"])
df = df[df["text_combined"].str.strip().str.len() > 0]
df = df.drop_duplicates(subset=["text_combined"]).reset_index(drop=True)
print("Shape after cleaning:", df.shape)


In [ ]:

# Class balance
label_counts = df["label"].value_counts()
print(label_counts)

plt.figure(figsize=(5, 4))
plt.bar(["Legitimate (0)", "Phishing (1)"], label_counts.sort_index().values, color=["#4C72B0", "#C44E52"])
plt.title("Class Distribution")
plt.ylabel("Number of emails")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120)
plt.show()


In [ ]:

# Email length distribution (in words)
df["word_count"] = df["text_combined"].astype(str).str.split().str.len()
print(df["word_count"].describe())

plt.figure(figsize=(6, 4))
plt.hist(df["word_count"].clip(upper=1000), bins=50, color="#55A868")
plt.title("Email Length Distribution (word count, clipped at 1000)")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("length_distribution.png", dpi=120)
plt.show()


## 3. Text Preprocessing

The text is already lowercased and lightly cleaned. We apply a small amount of additional normalisation
(stripping extra whitespace, removing stray non-alphanumeric characters) before tokenization.


In [ ]:

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " urltoken ", text)   # normalise URLs to a single token
    text = re.sub(r"[^a-z0-9\s]", " ", text)                   # strip punctuation/symbols
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text_combined"].apply(clean_text)
df[["text_combined", "clean_text"]].head(3)


## 4. Train / Validation / Test Split

The dataset is split 70% train / 15% validation / 15% test, stratified on the label to preserve class balance.


In [ ]:

X = df["clean_text"].values
y = df["label"].astype(int).values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Train label balance:", np.bincount(y_train) / len(y_train))
print("Val label balance:  ", np.bincount(y_val) / len(y_val))
print("Test label balance: ", np.bincount(y_test) / len(y_test))


## 5. Tokenization and Padding

We fit a `Tokenizer` on the training set only (to avoid data leakage), then convert all splits to padded
integer sequences of fixed length.


In [ ]:

VOCAB_SIZE = 20000
MAX_LEN = 250

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

X_train_pad = to_padded(X_train)
X_val_pad = to_padded(X_val)
X_test_pad = to_padded(X_test)

print("Padded shapes:", X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)

# Save tokenizer for later use in the desktop app / API
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


## 6. Model Architecture - Bidirectional LSTM

The model:
1. **Embedding layer** - learns a dense vector representation for each word in the vocabulary.
2. **Bidirectional LSTM** - reads the email text forwards and backwards, capturing sequential/contextual
   patterns (e.g. urgency phrases, suspicious call-to-action language) regardless of position in the email.
3. **Dropout** - regularisation to reduce overfitting.
4. **Dense (ReLU)** - a fully connected layer to combine learned features.
5. **Dense (Sigmoid)** - single output neuron for binary classification (phishing probability).


In [ ]:

EMBEDDING_DIM = 64

def build_model():
    model = models.Sequential([
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, mask_zero=True),
        layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
        layers.Dropout(0.4),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ]
    )
    return model

model = build_model()
model.summary()


## 7. Per-Epoch Metric Logging and Early Stopping

Keras logs loss, accuracy, precision, recall and AUC-ROC automatically each epoch via the metrics defined above.
We add a custom callback that additionally computes the **F1-score** on the validation set after every epoch
(since Keras has no built-in F1 metric), and prints a full metrics summary per iteration.

`EarlyStopping` monitors validation loss with a **patience of 3 epochs**: training stops once validation loss
fails to improve for 3 consecutive epochs, and the best weights are restored, preventing overfitting.


In [ ]:

class F1MetricLogger(callbacks.Callback):
    """Computes F1-score on the validation set at the end of every epoch and logs a full metrics summary."""
    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_probs = self.model.predict(self.X_val, verbose=0).ravel()
        val_preds = (val_probs >= 0.5).astype(int)
        f1 = f1_score(self.y_val, val_preds)
        logs["val_f1"] = f1

        record = {
            "epoch": epoch + 1,
            "loss": logs.get("loss"),
            "val_loss": logs.get("val_loss"),
            "accuracy": logs.get("accuracy"),
            "val_accuracy": logs.get("val_accuracy"),
            "precision": logs.get("precision"),
            "val_precision": logs.get("val_precision"),
            "recall": logs.get("recall"),
            "val_recall": logs.get("val_recall"),
            "auc": logs.get("auc"),
            "val_auc": logs.get("val_auc"),
            "val_f1": f1,
        }
        self.history.append(record)
        print(f"Epoch {epoch+1}: val_loss={logs.get('val_loss'):.4f} "
              f"val_accuracy={logs.get('val_accuracy'):.4f} "
              f"val_precision={logs.get('val_precision'):.4f} "
              f"val_recall={logs.get('val_recall'):.4f} "
              f"val_f1={f1:.4f} "
              f"val_auc={logs.get('val_auc'):.4f}")

f1_logger = F1MetricLogger(X_val_pad, y_val)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    "best_phishing_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)


## 8. Train the Model

In [ ]:

EPOCHS = 20
BATCH_SIZE = 128

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[f1_logger, early_stop, checkpoint],
    verbose=2
)


In [ ]:

# Save the full per-epoch metrics log (satisfies "logging of evaluation metrics after each iteration")
metrics_log = pd.DataFrame(f1_logger.history)
metrics_log.to_csv("epoch_metrics_log.csv", index=False)
metrics_log


## 9. Training Curves

In [ ]:

epochs_range = metrics_log["epoch"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Loss curve
axes[0, 0].plot(epochs_range, metrics_log["loss"], label="Train Loss", marker="o")
axes[0, 0].plot(epochs_range, metrics_log["val_loss"], label="Val Loss", marker="o")
axes[0, 0].set_title("Loss Curve")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()

# Accuracy curve
axes[0, 1].plot(epochs_range, metrics_log["accuracy"], label="Train Accuracy", marker="o")
axes[0, 1].plot(epochs_range, metrics_log["val_accuracy"], label="Val Accuracy", marker="o")
axes[0, 1].set_title("Accuracy Curve")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()

# Precision / Recall curve
axes[1, 0].plot(epochs_range, metrics_log["val_precision"], label="Val Precision", marker="o")
axes[1, 0].plot(epochs_range, metrics_log["val_recall"], label="Val Recall", marker="o")
axes[1, 0].plot(epochs_range, metrics_log["val_f1"], label="Val F1", marker="o")
axes[1, 0].set_title("Validation Precision / Recall / F1")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Score")
axes[1, 0].legend()

# AUC curve
axes[1, 1].plot(epochs_range, metrics_log["auc"], label="Train AUC", marker="o")
axes[1, 1].plot(epochs_range, metrics_log["val_auc"], label="Val AUC", marker="o")
axes[1, 1].set_title("AUC-ROC Curve (per epoch)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("AUC")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()


## 10. Final Evaluation on the Held-Out Test Set

The model (best weights, restored by early stopping) is now evaluated on the untouched test set.


In [ ]:

test_probs = model.predict(X_test_pad, verbose=0).ravel()
test_preds = (test_probs >= 0.5).astype(int)

test_accuracy = accuracy_score(y_test, test_preds)
test_precision = precision_score(y_test, test_preds)
test_recall = recall_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_auc = roc_auc_score(y_test, test_probs)

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test F1-score : {test_f1:.4f}")
print(f"Test AUC-ROC  : {test_auc:.4f}")
print()
print(classification_report(y_test, test_preds, target_names=["Legitimate", "Phishing"]))


In [ ]:

# Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize=(5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Phishing"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()


In [ ]:

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {test_auc:.4f})", color="#C44E52")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=120)
plt.show()


In [ ]:

# Precision-Recall Curve
precisions, recalls, _ = precision_recall_curve(y_test, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions, color="#55A868")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Test Set")
plt.tight_layout()
plt.savefig("pr_curve.png", dpi=120)
plt.show()


## 11. Save Model Artifacts

The trained model, tokenizer, and key configuration are saved to disk so they can be loaded by the
Flask/FastAPI backend and the Tkinter desktop application.


In [ ]:

model.save("phishing_bilstm_model.keras")

config = {
    "vocab_size": VOCAB_SIZE,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "test_accuracy": float(test_accuracy),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall),
    "test_f1": float(test_f1),
    "test_auc": float(test_auc),
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: phishing_bilstm_model.keras, tokenizer.pkl, model_config.json")
print(config)


## 12. Checking Model's Accuracy

Checking the model's predictions before using on the API & app.


In [ ]:

def predict_email(text):
    cleaned = clean_text(text)
    padded = to_padded([cleaned])
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Phishing" if prob >= 0.5 else "Legitimate"
    return label, float(prob)

samples = [
    "Dear customer, your account has been suspended. Click here immediately to verify your password and bank details or your account will be closed within 24 hours.",
    "Hi team, attached is the agenda for tomorrow's 10am sync. Let me know if you want to add anything.",
]

for s in samples:
    label, prob = predict_email(s)
    print(f"Prediction: {label} (phishing probability={prob:.4f})")
    print(f"Text: {s[:80]}...")
    print()


## 13. Explainable AI (XAI) with LIME — Bonus Feature

Deep learning text classifiers are effectively black boxes, a probability score alone doesn't tell an analyst
*why* an email was flagged. To make the model's decisions interpretable, **LIME** (Local Interpretable
Model-agnostic Explanations) was used. LIME explains an individual prediction by perturbing the input text (removing/
replacing words) many times, observing how the model's output probability changes, and fitting a simple
interpretable model locally to estimate which words pushed the prediction toward "Phishing" or "Legitimate".

This is valuable in a real SOC/mail-filtering context: an analyst reviewing a flagged email can see exactly
which words drove the decision, rather than trusting an opaque probability score.


In [ ]:

from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=["Legitimate", "Phishing"])

def lime_predict_proba(texts):
    """Wraps the model so LIME can call it like a standard scikit-learn predict_proba."""
    cleaned = [clean_text(t) for t in texts]
    padded = to_padded(cleaned)
    probs = model.predict(padded, batch_size=256, verbose=0).ravel()
    return np.vstack([1 - probs, probs]).T


def explain_email(text, num_features=10, num_samples=500):
    exp = explainer.explain_instance(
        text, lime_predict_proba, num_features=num_features, num_samples=num_samples
    )
    return exp

# Explain the two sanity-check emails from Section 12
for s in samples:
    exp = explain_email(s)
    label, prob = predict_email(s)
    print(f"Prediction: {label} (phishing probability={prob:.4f})")
    print(f"Text: {s[:90]}...")
    print("Top contributing words (positive = toward Phishing, negative = toward Legitimate):")
    for word, weight in exp.as_list():
        print(f"    {word:15s} {weight:+.4f}")
    print()


In [ ]:

# Visual explanation for the phishing example: word-level contribution bar chart
phishing_sample = samples[0]
exp = explain_email(phishing_sample)

words, weights = zip(*exp.as_list())
colors = ["#C44E52" if w > 0 else "#4C72B0" for w in weights]

plt.figure(figsize=(7, 5))
plt.barh(words, weights, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Contribution to 'Phishing' prediction")
plt.title("LIME Word-Level Explanation (Phishing Example)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("lime_explanation_phishing.png", dpi=120)
plt.show()


In [ ]:

# Save an interactive HTML explanation (openable in any browser) for the report/demo
exp_html = explain_email(samples[0])
exp_html.save_to_file("lime_explanation_sample.html")
print("Saved lime_explanation_sample.html")
